In [1]:
# Steps 1 and 2 in one place: get and clean the data, then define churn
import zipfile
from pathlib import Path

import pandas as pd
import requests

RAW = Path("../data/raw/online_retail_II.xlsx")
URL = "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"
END = pd.Timestamp("2011-12-01")  # December 2011 is incomplete

## Step 1: Get and clean the data

Download the UCI file, load both yearly sheets into one table, then apply the cleaning rules in order: drop rows without a customer ID (they can't be linked to a customer); drop exact duplicate rows; drop December 2011 (the file stops on 9 December, so the month is incomplete); drop non-product codes (postage, manual adjustments, bank charges, test items, gift vouchers); drop zero or negative prices. Finally, add `amount` (quantity × price) and split cancellations (invoice starting with "C") from purchases.

In [2]:
# Step 1a: download (skipped if the file is already there)
if not RAW.exists():
    RAW.parent.mkdir(parents=True, exist_ok=True)
    zip_path = RAW.with_suffix(".zip")
    zip_path.write_bytes(requests.get(URL, timeout=120).content)
    with zipfile.ZipFile(zip_path) as zf:
        RAW.write_bytes(zf.read(next(n for n in zf.namelist() if n.endswith(".xlsx"))))
    zip_path.unlink()

In [3]:
# Step 1b: load both yearly sheets into one table
sheets = pd.read_excel(RAW, sheet_name=None,
                       dtype={"Invoice": str, "StockCode": str, "Description": str, "Country": str})
raw = pd.concat(sheets.values(), ignore_index=True).rename(columns={
    "Invoice": "invoice", "StockCode": "stock_code", "Description": "description",
    "Quantity": "quantity", "InvoiceDate": "invoice_date", "Price": "price",
    "Customer ID": "customer_id", "Country": "country"})
len(raw)

1067371

In [4]:
# Step 1c: cleaning rules, counting rows after each one
NON_PRODUCT = {"POST", "DOT", "C2", "C3", "M", "D", "S", "B", "ADJUST", "ADJUST2", "BANK CHARGES",
               "PADS", "CRUK", "AMAZONFEE", "TEST001", "TEST002", "GIFT"}

steps = [("raw rows", len(raw))]
df = raw[raw["customer_id"].notna()]
steps.append(("no customer ID", len(df)))
df = df.drop_duplicates()
steps.append(("exact duplicates", len(df)))
df = df[df["invoice_date"] < END]
steps.append(("December 2011", len(df)))
code = df["stock_code"].str.strip().str.upper()
df = df[~(code.isin(NON_PRODUCT) | code.str.startswith("GIFT_"))]
steps.append(("non-products", len(df)))
df = df[df["price"] > 0]
steps.append(("price <= 0", len(df)))

report = pd.DataFrame(steps, columns=["step", "rows left"])
report["removed"] = -report["rows left"].diff().fillna(0).astype(int)
report

,step,rows left,removed
0,raw rows,1067371,0
1,no customer ID,824364,243007
2,exact duplicates,797885,26479
3,December 2011,780502,17383
4,non-products,776915,3587
5,price <= 0,776855,60


In [5]:
# Step 1d: add amount, split cancellations (invoice starts with "C") from purchases
df = df.assign(customer_id=df["customer_id"].astype("int64"),
               stock_code=df["stock_code"].str.strip(),
               amount=df["quantity"] * df["price"])
is_cancel = df["invoice"].str.upper().str.startswith("C")
purchases, cancellations = df[~is_cancel], df[is_cancel]
len(purchases), len(cancellations)

(759615, 17240)

## Step 2: Define churn

Churn is measured at a series of cutoff dates (the first of each month). At each cutoff, the population is every customer with at least one purchase in the previous 365 days, and a customer is labelled churned (1) if they make no purchase in the following 90 days (60 and 120 days as checks). Only purchases before the cutoff may be used to describe a customer. Each cutoff needs 365 days of history before it and a complete label window after it, which leaves 10 cutoffs at 90 days (December 2010 to September 2011). The last 3 are for testing; training uses only cutoffs whose label window closes before testing begins, so May and June 2011 are skipped.

In [6]:
# Step 2a: churn label at one cutoff date
def snapshot(purchases, cutoff, horizon=90, active=365):
    d = purchases["invoice_date"]
    active_ids = purchases.loc[(d >= cutoff - pd.Timedelta(days=active)) & (d < cutoff), "customer_id"].unique()
    returned = purchases.loc[(d >= cutoff) & (d < cutoff + pd.Timedelta(days=horizon)), "customer_id"].unique()
    return pd.DataFrame({"customer_id": active_ids, "cutoff": cutoff, "horizon": horizon,
                         "churn": (~pd.Series(active_ids).isin(returned)).astype(int)})

# month starts with 365 days of history before and a full label window after
def cutoffs(horizon):
    return pd.date_range(pd.Timestamp("2009-12-01") + pd.Timedelta(days=365),
                         END - pd.Timedelta(days=horizon), freq="MS")

In [7]:
# Step 2b: training and test cutoffs (90 days); training labels must end before testing starts
cuts = list(cutoffs(90))
test = cuts[-3:]
train = [c for c in cuts[:-3] if c + pd.Timedelta(days=90) <= test[0]]
[c.strftime("%Y-%m") for c in train], [c.strftime("%Y-%m") for c in test]

(['2010-12', '2011-01', '2011-02', '2011-03', '2011-04'],
 ['2011-07', '2011-08', '2011-09'])

In [8]:
# Step 2c: churn rates for every cutoff and window
snaps = pd.concat([snapshot(purchases, c, h) for h in (60, 90, 120) for c in cutoffs(h)], ignore_index=True)
s90 = snaps[snaps["horizon"] == 90]
print(f"{len(s90):,} snapshots, {s90['customer_id'].nunique():,} customers, "
      f"churn {s90['churn'].mean():.1%} (train {s90[s90['cutoff'].isin(train)]['churn'].mean():.1%}, "
      f"test {s90[s90['cutoff'].isin(test)]['churn'].mean():.1%})")
snaps.pivot_table(index="cutoff", columns="horizon", values="churn", aggfunc="mean").round(3)

42,850 snapshots, 5,224 customers, churn 61.8% (train 64.9%, test 56.2%)


horizon,60,90,120
cutoff,,,
2010-12-01,0.733,0.668,0.607
2011-01-01,0.749,0.669,0.609
2011-02-01,0.728,0.654,0.594
2011-03-01,0.711,0.631,0.576
2011-04-01,0.701,0.623,0.574
2011-05-01,0.681,0.613,0.575
2011-06-01,0.693,0.634,0.562
2011-07-01,0.713,0.614,0.530
2011-08-01,0.672,0.569,0.477
